# Retrieve and Clean Student CSV Datasets

This notebook loads the yearly student CSV files from `DataSets/`, standardizes their columns, combines them into a single dataset, and produces a cleaned version.

Run the cells in order from top to bottom. It assumes the notebook's working directory is `Scripts/`.

In [1]:
import csv
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd().resolve().parent
DATASETS_DIR = BASE_DIR / "DataSets"

## Helper Functions

Discover CSV files, detect their delimiter, and standardize column names.

In [2]:
def get_csv_files():
    csv_files = sorted(DATASETS_DIR.glob("*.csv"))

    if not csv_files:
        print(f"No CSV files found in {DATASETS_DIR}")
        return []

    return csv_files


def detect_delimiter(path: Path):
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        sample = file.read(4096)
        file.seek(0)

    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=",;")
        return dialect.delimiter
    except csv.Error:
        return ";" if ";" in sample else ","


def standardize_column_name(column_name):
    return str(column_name).strip().lower().replace(" ", "_")


def get_standardized_columns(csv_file: Path):
    delimiter = detect_delimiter(csv_file)

    with csv_file.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file, delimiter=delimiter)
        header = next(reader, None)

    if header is None:
        return []

    standardized = [standardize_column_name(column) for column in header]
    return standardized


def load_csv_as_dataframe(csv_file: Path):
    delimiter = detect_delimiter(csv_file)
    df = pd.read_csv(csv_file, sep=delimiter, encoding="utf-8-sig")
    df.columns = [standardize_column_name(col) for col in df.columns]
    return df

## Preview Raw Files

Print the columns and first rows of each raw CSV file.

In [3]:
def print_first_rows(csv_file: Path, rows_to_show: int = 5):
    delimiter = detect_delimiter(csv_file)

    with csv_file.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file, delimiter=delimiter)
        rows = list(reader)

    if not rows:
        print(f"\n=== {csv_file.name} ===")
        print("File is empty.")
        return

    print(f"\n=== {csv_file.name} ===")
    print("Columns:", get_standardized_columns(csv_file))
    for row in rows[:rows_to_show]:
        print(row)


csv_files = get_csv_files()

for csv_file in csv_files:
    print_first_rows(csv_file)


=== datos_estudiantes_2019.csv ===
Columns: ['id_persona', 'sexo', 'rol', 'departamento', 'subsistema', 'ciclo', 'grado', 'zona', 'contexto', 'año_lectivo', 'cantidad_de_días_ingreso_a_crea', 'cantidad_de_entregas_de_tareas', 'cantidad_de_comentarios_posteados', 'cantidad_de_acciones_totales', 'cantidad_de_días_de_ingreso_a_matific', 'cantidad_de_episodios_finalizados_en_matific', 'cantidad_de_días_de_ingreso_a_pam', 'cantidad_de_actividades_finalizadas_en_pam', 'cantidad_de_días_de_ingreso_a_biblioteca', 'cantidad_de_préstamos_en_biblioteca']
['Id persona', 'Sexo', 'Rol', 'Departamento', 'Subsistema', 'Ciclo', 'Grado', 'Zona', 'Contexto', 'Año lectivo', 'Cantidad de días ingreso a CREA', 'Cantidad de entregas de tareas', 'Cantidad de Comentarios posteados', 'Cantidad de Acciones totales', 'Cantidad de días de ingreso a Matific', 'Cantidad de episodios finalizados en Matific', 'Cantidad de días de ingreso a PAM', 'Cantidad de actividades finalizadas en PAM', 'Cantidad de días de ingre

## Combine Datasets

Load every yearly CSV, standardize columns, concatenate, drop duplicates, and save the combined dataset.

> Note: the combined and cleaned outputs are large (500MB+) and are excluded from git via `.gitignore`.

In [4]:
def create_combined_dataset(output_name: str = "datos_estudiantes_total.csv"):
    csv_files = get_csv_files()
    if not csv_files:
        return None

    dataframes = [load_csv_as_dataframe(csv_file) for csv_file in csv_files]
    combined = pd.concat(dataframes, ignore_index=True)
    combined = combined.drop_duplicates()

    output_path = DATASETS_DIR / output_name
    combined.to_csv(output_path, index=False)

    print(f"Combined dataset saved to: {output_path}")
    print(f"Rows: {len(combined)}")
    print(f"Columns: {list(combined.columns)}")
    return combined


combined = create_combined_dataset()

Combined dataset saved to: /Users/gerardo/Documents/GitHub/PlanCeibal-UTEC26-MachineLearning/DataSets/datos_estudiantes_total.csv
Rows: 9120882
Columns: ['id_persona', 'sexo', 'rol', 'departamento', 'subsistema', 'ciclo', 'grado', 'zona', 'contexto', 'año_lectivo', 'cantidad_de_días_ingreso_a_crea', 'cantidad_de_entregas_de_tareas', 'cantidad_de_comentarios_posteados', 'cantidad_de_acciones_totales', 'cantidad_de_días_de_ingreso_a_matific', 'cantidad_de_episodios_finalizados_en_matific', 'cantidad_de_días_de_ingreso_a_pam', 'cantidad_de_actividades_finalizadas_en_pam', 'cantidad_de_días_de_ingreso_a_biblioteca', 'cantidad_de_préstamos_en_biblioteca', 'cantidad_de_entregas_de_tareas_en_crea', 'cantidad_de_comentarios_posteados_en_crea', 'cantidad_de_acciones_totales_en_crea']


## Clean the Combined Dataset

Normalize text, replace known "missing" placeholders with NA, and coerce numeric-looking columns. Missing values are preserved so model preprocessing can be fit on training data only.

In [ ]:
def clean_combined_dataset(df: pd.DataFrame):
    cleaned = df.copy()
    cleaned = cleaned.drop_duplicates()

    for column in cleaned.columns:
        if cleaned[column].dtype == "object":
            cleaned[column] = cleaned[column].astype(str).str.strip().str.lower()
            cleaned[column] = cleaned[column].replace(
                {
                    "": pd.NA,
                    "na": pd.NA,
                    "n/a": pd.NA,
                    "nan": pd.NA,
                    "null": pd.NA,
                    "none": pd.NA,
                    "sin dato": pd.NA,
                    "sin_dato": pd.NA,
                    "sin-dato": pd.NA,
                    "unknown": pd.NA,
                    "desconocido": pd.NA,
                }
            )

    if "id_persona" in cleaned.columns:
        cleaned = cleaned.dropna(subset=["id_persona"])

    for column in cleaned.columns:
        if cleaned[column].dtype == "object":
            numeric_values = pd.to_numeric(cleaned[column], errors="coerce")
            valid_ratio = numeric_values.notna().sum() / max(cleaned[column].notna().sum(), 1)
            if valid_ratio > 0.8:
                cleaned[column] = numeric_values

    for column in cleaned.columns:
        if pd.api.types.is_numeric_dtype(cleaned[column]):
            if cleaned[column].dropna().mod(1).eq(0).all():
                cleaned[column] = cleaned[column].astype("Int64")

    return cleaned


if combined is not None:
    cleaned = clean_combined_dataset(combined)
    output_path = DATASETS_DIR / "datos_estudiantes_total_clean.csv"
    cleaned.to_csv(output_path, index=False)
    print(f"Clean dataset saved to: {output_path}")
    print(f"Cleaned rows: {len(cleaned)}")
    print(f"Cleaned columns: {list(cleaned.columns)}")

Clean dataset saved to: /Users/gerardo/Documents/GitHub/PlanCeibal-UTEC26-MachineLearning/DataSets/datos_estudiantes_total_clean.csv
Cleaned rows: 9120882
Cleaned columns: ['id_persona', 'sexo', 'rol', 'departamento', 'subsistema', 'ciclo', 'grado', 'zona', 'contexto', 'año_lectivo', 'cantidad_de_días_ingreso_a_crea', 'cantidad_de_entregas_de_tareas', 'cantidad_de_comentarios_posteados', 'cantidad_de_acciones_totales', 'cantidad_de_días_de_ingreso_a_matific', 'cantidad_de_episodios_finalizados_en_matific', 'cantidad_de_días_de_ingreso_a_pam', 'cantidad_de_actividades_finalizadas_en_pam', 'cantidad_de_días_de_ingreso_a_biblioteca', 'cantidad_de_préstamos_en_biblioteca', 'cantidad_de_entregas_de_tareas_en_crea', 'cantidad_de_comentarios_posteados_en_crea', 'cantidad_de_acciones_totales_en_crea']
